# Exercise 4 — OpsGuardrails

`OpsGuardrails` is the Day-87 guardrail trio — Guard, BudgetTracker, ApprovalGate — bundled into a single ops-specific config object.  `check_budget()` both checks *and* records in one call, so the loop only needs to call it once per iteration.

In [ ]:
import json
from dataclasses import dataclass, field

def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    text = str(text)
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except Exception:
        return None
def _validate_text(text, max_length=None, banned=None):
    text_str = str(text)
    if max_length is not None and len(text_str) > max_length:
        return False, "text exceeds max_length (" + str(len(text_str)) + " > " + str(max_length) + " chars)"
    if banned:
        lower = text_str.lower()
        for p in banned:
            if str(p).lower() in lower: return False, "banned pattern found: " + repr(p)
    return True, ""

class _Guard:
    def __init__(self, max_length=None, banned=None):
        self.max_length = max_length; self.banned = list(banned) if banned else []
    def check(self, text): return _validate_text(text, self.max_length, self.banned)

class _ApprovalGate:
    def __init__(self, approve_fn=None):
        self._fn = approve_fn if approve_fn is not None else (lambda a: True)
    def check(self, action):
        try: result = bool(self._fn(str(action)))
        except Exception: result = False
        return (True, "approved") if result else (False, "rejected by approval gate")

class _BudgetTracker:
    def __init__(self, max_calls=None):
        self.max_calls = max_calls; self._count = 0
    def ok(self):
        if self.max_calls is not None and self._count >= self.max_calls:
            return False, "budget exceeded (" + str(self._count) + "/" + str(self.max_calls) + " calls)"
        return True, ""
    def record(self): self._count += 1
    def reset(self): self._count = 0
    @property
    def count(self): return self._count


# ── Exercise: implement OpsGuardrails ────────────────────────────────────────

class OpsGuardrails:
    """Three guardrail layers bundled for the OpsAgent.

    max_budget    : max ReAct iterations across all run() calls
    banned_inputs : list of strings blocked at the input boundary
    approve_fn    : callable(goal: str) -> bool  (default: auto-approve)
    """

    def __init__(self, max_budget=20, banned_inputs=None, approve_fn=None):
        # TODO: build self._input_guard = _Guard(max_length=500, banned=...)
        # self._budget = _BudgetTracker(max_calls=max_budget)
        # self._gate   = _ApprovalGate(approve_fn=approve_fn)
        pass

    def check_input(self, query):
        # TODO: return self._input_guard.check(str(query))
        return True, ""

    def check_budget(self):
        # TODO: call self._budget.ok(); if ok record and return (True, "")
        # if not ok return (False, reason) WITHOUT recording
        return True, ""

    def check_approval(self, goal):
        # TODO: return self._gate.check(str(goal))
        return True, "approved"

    def reset(self):
        # TODO: self._budget.reset()
        pass

    @property
    def budget_count(self):
        # TODO: return self._budget.count
        return 0


### Checks

In [ ]:
checks = 0

# 1 — OpsGuardrails constructs; default auto-approves everything
try:
    g = OpsGuardrails(max_budget=10)
    ok, _ = g.check_input("Run all checks")
    assert ok
    approved, _ = g.check_approval("any goal")
    assert approved
    checks += 1; print("✅ 1 OpsGuardrails constructs and auto-approves by default")
except Exception as e:
    print("❌ 1:", e)

# 2 — check_input blocks a banned pattern
try:
    g = OpsGuardrails(banned_inputs=["rm -rf", "drop table"])
    ok, reason = g.check_input("please rm -rf /")
    assert not ok and "rm -rf" in reason.lower()
    ok2, _ = g.check_input("safe query")
    assert ok2
    checks += 1; print("✅ 2 check_input blocks banned patterns")
except Exception as e:
    print("❌ 2:", e)

# 3 — check_budget allows calls under the limit and records
try:
    g = OpsGuardrails(max_budget=3)
    ok1, _ = g.check_budget(); assert ok1 and g.budget_count == 1
    ok2, _ = g.check_budget(); assert ok2 and g.budget_count == 2
    ok3, _ = g.check_budget(); assert ok3 and g.budget_count == 3
    checks += 1; print("✅ 3 check_budget allows calls and increments count")
except Exception as e:
    print("❌ 3:", e)

# 4 — check_budget blocks when limit reached
try:
    g = OpsGuardrails(max_budget=2)
    g.check_budget(); g.check_budget()  # use up budget
    ok, reason = g.check_budget()
    assert not ok and ("budget" in reason.lower() or "exceeded" in reason.lower())
    assert g.budget_count == 2   # count did NOT increment on the blocked call
    checks += 1; print("✅ 4 check_budget blocks at limit; count not incremented on block")
except Exception as e:
    print("❌ 4:", e)

# 5 — check_approval blocks when approve_fn returns False; reset works
try:
    g = OpsGuardrails(max_budget=5, approve_fn=lambda goal: False)
    approved, reason = g.check_approval("any goal")
    assert not approved
    g.reset(); assert g.budget_count == 0
    checks += 1; print("✅ 5 check_approval blocks with reject fn; reset() clears count")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
